# Medical Chatbot - Research Notebook
This notebook uses modular imports from `src/helper.py`, `src/prompt.py`, and logic from `store_index.py`.

## 1. Setup — Add Project Root to Path

In [1]:
import sys
import os

# we go one level up to project root,because notebook inside /reserach
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
  sys.path.insert(0, project_root)

print("Project root: ", project_root)
print("sys.path[0]: ", sys.path[0])

Project root:  d:\Project\Medical-Chatbot\end-to-end-medical-chatbot-generative-ai
sys.path[0]:  d:\Project\Medical-Chatbot\end-to-end-medical-chatbot-generative-ai


## 2. Load Environment Variables

In [2]:
from dotenv import load_dotenv

load_dotenv()

PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')

assert PINECONE_API_KEY, "PINECONE_API_KEY not found in .env"
assert GROQ_API_KEY, "GROQ_API_KEY not found in .env"

print("Environment variables loaded successfully.")

Environment variables loaded successfully.


## 3. Import Helpers from `src/helper.py`

In [3]:
from src.helper import load_pdf_file, text_split, download_hugging_face_embeddings

print("Helpers imported successfully.")

Helpers imported successfully.


## 4. Load & Chunk PDF Data

In [4]:
# pdf inside data so load it
data_path = os.path.join(project_root, "Data")

extracted_data = load_pdf_file(data=data_path)
print(f"Total documents loaded: {len(extracted_data)}")

Total documents loaded: 637


In [5]:
text_chunks = text_split(extracted_data)
print(f"Total text chunks: {len(text_chunks)}")

Total text chunks: 5860


## 5. Download Embeddings

In [6]:
embedddings = download_hugging_face_embeddings()

sample = embedddings.embed_query("Hello world")
print(f"Embedding dimension: {len(sample)}")

d:\Project\Medical-Chatbot\end-to-end-medical-chatbot-generative-ai\src\helper.py:24: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


Embedding dimension: 384


## 6. Pinecone Vector Store Setup

In [7]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "medicalbot"

if not pc.has_index(index_name):
  pc.create_index(
    name=index_name,
    dimension=384,
    metric="consine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
  )
  print("Index created - uploading embedddings...")
  docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embedddings,
  )
  print("Embeddings uplaoded!")
else:
  print("Index already exists - loading...")
  docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedddings,
  )
  print("Existing index loaded!")

Index already exists - loading...
Existing index loaded!


## 7. Build Retriever

In [8]:
retriever = docsearch.as_retriever(
  search_type="similarity",
  search_kwargs={"k": 3}
)

# test retrieval
text_docs = retriever.invoke("What is Acne?")
print(f"Retrieved {len(text_docs)} documents for 'What is Acne?'")
print("\nFirst result preview: ")
print(text_docs[0].page_content[:300])

Retrieved 3 documents for 'What is Acne?'

First result preview: 
GALE ENCYCLOPEDIA OF MEDICINE 226
Acne
GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26


## 8. Load Prompt from `src/prompt.py`

In [9]:
from src.prompt import system_prompt
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
  ("system", system_prompt),
  ("human", "{input}"),
])

print("Prompt loaded from src/prompt.py")
print("\nSystem prompt preview: ")
print(system_prompt[:200])

Prompt loaded from src/prompt.py

System prompt preview: 
You are an assistant for question-answering tasks.Use the following pieces of retrieved context to answerthe question. If you don't know the answer, say that you don't know. Use three sentences maximu


## 9. Initialize LLM & Build RAG Chain

In [10]:
from langchain_groq import ChatGroq
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

llm = ChatGroq(
  temperature=0.4,
  max_tokens=500,
  model_name="llama-3.3-70b-versatile"
)

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("RAG chain ready.")

RAG chain ready.


## 10. Test Queries

In [11]:
response = rag_chain.invoke({"input": "What is Acoustic neuroma?"})
print("Q: What is Acoustic neuroma?")
print("A:", response["answer"])

Q: What is Acoustic neuroma?
A: An acoustic neuroma is a benign tumor involving cells of the myelin sheath that surrounds the vestibulocochlear nerve, also known as the eighth cranial nerve. It affects the nerve that extends from the inner ear to the brain, which is responsible for hearing and balance. The tumor is non-cancerous and typically grows slowly.


In [12]:
# something outside the medical context
response = rag_chain.invoke({"input": "What is stats?"})
print("Q: What is stats?")
print("A:", response["answer"])

Q: What is stats?
A: I don't know what "stats" refers to in this context, as it is not clearly defined in the provided text. The text appears to discuss medical topics, including blood counts and heart conditions, but does not mention "stats" explicitly. If you have more context or information, I may be able to help further.


In [13]:
response = rag_chain.invoke({"input": "What is Acne and how is it treated?"})
print("Q: What is Acne and how is it treated?")
print("A:", response["answer"])

Q: What is Acne and how is it treated?
A: Acne is a skin disorder in which the sebaceous glands become inflamed. I don't have information on specific treatments in the provided context, but it mentions various articles and journals discussing acne treatment. The context suggests that there are various treatments available, but the details are not provided.
